In [27]:
#importlar
import os
import cv2
import numpy as np
import scipy.io
from joblib import load
from skimage.feature import hog
from sklearn.decomposition import PCA
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


In [28]:
class AgeGenderCNN(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super(AgeGenderCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)
        )

        # Flatten sonrası boyutu otomatik hesapla
        dummy_input = torch.zeros(1, 1, *input_size)
        dummy_output = self.features(dummy_input)
        self.flattened_size = dummy_output.view(1, -1).shape[1]

        # Cinsiyet tahmini başlığı
        self.gender_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

        # Yaş tahmini başlığı
        self.age_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        gender_out = torch.sigmoid(self.gender_head(x))
        age_out = self.age_head(x)
        return gender_out, age_out


In [29]:
class AgeGenderDataset(Dataset):
    def __init__(self, X, y_gender, y_age):
        self.X = X.astype(np.float32) / 255.0
        self.y_gender = y_gender.astype(np.float32).reshape(-1, 1)
        self.y_age = y_age.astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)
        return torch.tensor(x), torch.tensor(self.y_gender[idx]), torch.tensor(self.y_age[idx])

In [30]:
def extract_hog_features(images):
    feats = []
    for img in images:
        h = hog(img,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                feature_vector=True)
        feats.append(h)
    return np.array(feats)

In [31]:
#Method 1 için test
def test_model_1(X_test, y_age_test, y_gender_test, name='dataset'):
    model_dir = f'models/model1_3/'

    # Model ve PCA'yı yükle
    pca = load(f'{model_dir}pca_{name}.joblib')
    X_hog = extract_hog_features(X_test)
    X_pca = pca.transform(X_hog)

    print(f"\n=== [ {name.upper()} ] TEST SONUÇLARI ===")

    # --- SVM + SVR ---
    svm = load(f'{model_dir}svm_gender_{name}.joblib')
    svr = load(f'{model_dir}svr_age_{name}.joblib')
    pred_gender_svm = svm.predict(X_pca)
    pred_age_svr = svr.predict(X_pca)

    print(f"[SVM] Gender Accuracy : {accuracy_score(y_gender_test, pred_gender_svm):.3f}")
    print(f"[SVM] Precision       : {precision_score(y_gender_test, pred_gender_svm, zero_division=0):.3f}")
    print(f"[SVM] Recall          : {recall_score(y_gender_test, pred_gender_svm, zero_division=0):.3f}")
    print(f"[SVM] F1-Score        : {f1_score(y_gender_test, pred_gender_svm, zero_division=0):.3f}")
    print(f"[SVR] Age MAE         : {mean_absolute_error(y_age_test, pred_age_svr):.2f}")
    print(f"[SVR] Age RMSE        : {root_mean_squared_error(y_age_test, pred_age_svr):.2f}")

    # --- MLP ---
    mlp_clf = load(f'{model_dir}mlp_gender_{name}.joblib')
    mlp_reg = load(f'{model_dir}mlp_age_{name}.joblib')
    pred_gender_mlp = mlp_clf.predict(X_pca)
    pred_age_mlp = mlp_reg.predict(X_pca)

    print(f"\n[MLP] Gender Accuracy : {accuracy_score(y_gender_test, pred_gender_mlp):.3f}")
    print(f"[MLP] Precision       : {precision_score(y_gender_test, pred_gender_mlp, zero_division=0):.3f}")
    print(f"[MLP] Recall          : {recall_score(y_gender_test, pred_gender_mlp, zero_division=0):.3f}")
    print(f"[MLP] F1-Score        : {f1_score(y_gender_test, pred_gender_mlp, zero_division=0):.3f}")
    print(f"[MLP] Age MAE         : {mean_absolute_error(y_age_test, pred_age_mlp):.2f}")
    print(f"[MLP] Age RMSE        : {root_mean_squared_error(y_age_test, pred_age_mlp):.2f}")

    # --- Random Forest ---
    rf_clf = load(f'{model_dir}rf_gender_{name}.joblib')
    # rf_reg = load(f'{model_dir}rf_age_{name}.joblib')
    pred_gender_rf = rf_clf.predict(X_pca)
    # pred_age_rf = rf_reg.predict(X_pca)

    print(f"\n[RF]  Gender Accuracy : {accuracy_score(y_gender_test, pred_gender_rf):.3f}")
    print(f"[RF]  Precision       : {precision_score(y_gender_test, pred_gender_rf, zero_division=0):.3f}")
    print(f"[RF]  Recall          : {recall_score(y_gender_test, pred_gender_rf, zero_division=0):.3f}")
    print(f"[RF]  F1-Score        : {f1_score(y_gender_test, pred_gender_rf, zero_division=0):.3f}")
    # print(f"[RF]  Age MAE         : {mean_absolute_error(y_age_test, pred_age_rf):.2f}")
    # print(f"[RF]  Age RMSE        : {root_mean_squared_error(y_age_test, pred_age_rf):.2f}")

In [32]:
#Method 2 için test
def test_model_2(model, dataloader, device):
    model.eval()
    all_preds_g = []
    all_preds_a = []
    all_true_g = []
    all_true_a = []

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x = x.to(device)
            pred_g, pred_a = model(x)
            all_preds_g += pred_g.cpu().numpy().flatten().tolist()
            all_preds_a += pred_a.cpu().numpy().flatten().tolist()
            all_true_g += y_g.numpy().flatten().tolist()
            all_true_a += y_a.numpy().flatten().tolist()

    pred_g_bin = [1 if p > 0.5 else 0 for p in all_preds_g]

    print("\n=== [TEST SONUÇLARI] ===")
    print(f"Cinsiyet - Accuracy : {accuracy_score(all_true_g, pred_g_bin):.3f}")
    print(f"Cinsiyet - Precision: {precision_score(all_true_g, pred_g_bin, zero_division=0):.3f}")
    print(f"Cinsiyet - Recall   : {recall_score(all_true_g, pred_g_bin, zero_division=0):.3f}")
    print(f"Cinsiyet - F1-score : {f1_score(all_true_g, pred_g_bin, zero_division=0):.3f}")

    print(f"Yaş - MAE           : {mean_absolute_error(all_true_a, all_preds_a):.2f}")
    print(f"Yaş - RMSE          : {root_mean_squared_error(all_true_a, all_preds_a):.2f}")

In [33]:
def predict_from_folder(folder_path, img_size=(128, 128), model_type='hog', name='UTKface_1'):
    images = []
    true_ages = []
    true_genders = []
    filenames = []

    for fname in os.listdir(folder_path):
        if fname.endswith(('.jpg', '.png', '.jpeg')):
            img = cv2.imread(os.path.join(folder_path, fname), cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, img_size)
            images.append(img)

            parts = fname.split('_')
            true_ages.append(int(parts[0]))
            true_genders.append(int(parts[1]))
            filenames.append(fname)

    X = np.array(images)

    if model_type == 'hog':
        test_model_1(X, true_ages, true_genders, name)
    else:
        dataset = AgeGenderDataset(X, np.array(true_genders), np.array(true_ages))
        dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = AgeGenderCNN().to(device)
        model.load_state_dict(torch.load("best_model_utk6.pth"))
        test_model_2(model, dataloader, device)


In [34]:
def predict_single_image(image_path, img_size=(128, 128), model_type='hog', name='UTKface_1'):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, img_size)

    parts = os.path.basename(image_path).split('_')
    true_age = int(parts[0])
    true_gender = int(parts[1])

    if model_type == 'hog':
        pca = load(f'models/model1_3/pca_{name}.joblib')
        svm = load(f'models/model1_3/svm_gender_{name}.joblib')
        svr = load(f'models/model1_3/svr_age_{name}.joblib')

        feat = extract_hog_features([img])
        feat_pca = pca.transform(feat)

        pred_gender = svm.predict(feat_pca)[0]
        pred_age = svr.predict(feat_pca)[0]

    else:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = AgeGenderCNN().to(device)
        model.load_state_dict(torch.load("best_model_utk6.pth"))
        model.eval()

        x = img.astype(np.float32) / 255.0
        x = torch.tensor(x).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            pred_g, pred_a = model(x)
            pred_gender = int(pred_g.item() > 0.5)
            pred_age = pred_a.item()

    print(f"Gerçek: Yaş={true_age}, Cinsiyet={true_gender}")
    print(f"Tahmin: Yaş={round(pred_age)}, Cinsiyet={pred_gender}")


In [35]:
# Tüm klasör için demo:
predict_from_folder("demo_samples/", model_type="hog", name="UTKface_1")

# Tek görsel için:
predict_single_image("demo_samples/20_0_0_20170113132609281.jpg", model_type="hog")



=== [ UTKFACE_1 ] TEST SONUÇLARI ===
[SVM] Gender Accuracy : 1.000
[SVM] Precision       : 1.000
[SVM] Recall          : 1.000
[SVM] F1-Score        : 1.000
[SVR] Age MAE         : 6.41
[SVR] Age RMSE        : 9.25

[MLP] Gender Accuracy : 1.000
[MLP] Precision       : 1.000
[MLP] Recall          : 1.000
[MLP] F1-Score        : 1.000
[MLP] Age MAE         : 2.51
[MLP] Age RMSE        : 3.49

[RF]  Gender Accuracy : 1.000
[RF]  Precision       : 1.000
[RF]  Recall          : 1.000
[RF]  F1-Score        : 1.000
Gerçek: Yaş=20, Cinsiyet=0
Tahmin: Yaş=20, Cinsiyet=0


In [36]:
# Tüm klasör için demo:
predict_from_folder("demo_samples/", model_type="cnn", name="UTKface_1")

# Tek görsel için:
predict_single_image("demo_samples/20_0_0_20170113132609281.jpg", model_type="cnn")



=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 1.000
Cinsiyet - Precision: 1.000
Cinsiyet - Recall   : 1.000
Cinsiyet - F1-score : 1.000
Yaş - MAE           : 3.67
Yaş - RMSE          : 5.24
Gerçek: Yaş=20, Cinsiyet=0
Tahmin: Yaş=19, Cinsiyet=0


In [37]:
X_test = np.load("X_test_demo_imdbwiki.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_demo_imdbwiki.npy")
y_gen_test = np.load("y_gender_test_demo_imdbwiki.npy")

In [38]:
test_model_1(X_test, y_age_test, y_gen_test, "imdb_wiki_1")


=== [ IMDB_WIKI_1 ] TEST SONUÇLARI ===
[SVM] Gender Accuracy : 0.875
[SVM] Precision       : 0.857
[SVM] Recall          : 1.000
[SVM] F1-Score        : 0.923
[SVR] Age MAE         : 16.50
[SVR] Age RMSE        : 19.70

[MLP] Gender Accuracy : 0.875
[MLP] Precision       : 1.000
[MLP] Recall          : 0.833
[MLP] F1-Score        : 0.909
[MLP] Age MAE         : 13.28
[MLP] Age RMSE        : 19.08

[RF]  Gender Accuracy : 0.875
[RF]  Precision       : 0.857
[RF]  Recall          : 1.000
[RF]  F1-Score        : 0.923


In [39]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderCNN(input_size=(64, 64)).to(device)
model.load_state_dict(torch.load("best_model_imdbwiki6pth"))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model_2(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.875
Cinsiyet - Precision: 0.857
Cinsiyet - Recall   : 1.000
Cinsiyet - F1-score : 0.923
Yaş - MAE           : 11.81
Yaş - RMSE          : 16.84
